In [2]:
import pandas as pd
import json
import os

print("🔍 BẮT ĐẦU CHỤP X-QUANG DỮ LIỆU THÔ (RAW DATA PROFILING)...\n")
base_path = "../data/1_bronze"

# =========================================================================
# 1. NẠP DỮ LIỆU THÔ ĐỂ DÒ LỖI (KHÔNG LÀM SẠCH)
# =========================================================================
# Chỉ đọc 100k dòng transactions để khảo sát nhanh, tránh treo RAM
df_trans_raw = pd.read_csv(os.path.join(base_path, "transactions_data.csv"), nrows=100000)
df_users_raw = pd.read_csv(os.path.join(base_path, "users_data.csv"))
df_cards_raw = pd.read_csv(os.path.join(base_path, "cards_data.csv"))

with open(os.path.join(base_path, "mcc_codes.json"), "r", encoding="utf-8") as f:
    mcc_dict = json.load(f)
df_mcc_raw = pd.DataFrame(list(mcc_dict.items()), columns=['mcc', 'merchant_category'])

with open(os.path.join(base_path, "train_fraud_labels.json"), "r", encoding="utf-8") as f:
    fraud_dict = json.load(f)
df_labels_raw = pd.DataFrame(list(fraud_dict.items()), columns=['transaction_id', 'is_fraud'])

datasets = {
    "Bảng Transactions (Thô)": df_trans_raw,
    "Bảng Users Profiles (Thô)": df_users_raw,
    "Bảng Cards Profiles (Thô)": df_cards_raw,
    "Bảng Danh mục MCC (Thô)": df_mcc_raw,
    "Bảng Nhãn Fraud (Thô)": df_labels_raw
}

# =========================================================================
# 2. MÁY QUÉT TỰ ĐỘNG: TÌM LỖ KHUYẾT THIẾU & TRÙNG LẶP
# =========================================================================
for name, df in datasets.items():
    print("="*60)
    print(f"📊 BÁO CÁO DÒ LỖI: {name.upper()}")
    print("="*60)
    
    print(f"🔹 Kích thước: {df.shape[0]} dòng, {df.shape[1]} cột")
    
    # Dò lỗi Khuyết thiếu (Missing Values)
    missing_data = df.isna().sum()
    print("\n⚠️ Lỗi Khuyết thiếu (NaN):")
    if missing_data.sum() > 0:
        missing_df = pd.DataFrame({'Số dòng trống': missing_data, 'Tỷ lệ (%)': (missing_data / len(df)) * 100})
        print(missing_df[missing_df['Số dòng trống'] > 0])
    else:
        print("👉 Sạch sẽ: Không có cột nào bị rỗng.")
        
    # Dò lỗi Trùng lặp (Duplicates)
    print("\n⚠️ Lỗi Trùng lặp (Duplicates):")
    try:
        dup_count = df.duplicated().sum()
        print(f"👉 Số dòng bị trùng lặp hoàn toàn: {dup_count}")
    except TypeError:
        print("👉 (Bỏ qua do chứa cấu trúc phức tạp unhashable)")
        
    # In ra kiểu dữ liệu để phát hiện lỗi định dạng (VD: số nhưng bị nhận là chữ)
    print("\n⚠️ Kiểu dữ liệu (Dtypes) đang bị nhận diện:")
    print(df.dtypes)
    print("\n" + "-"*60 + "\n")

# =========================================================================
# 3. KÍNH HIỂN VI: SOI SÂU VÀO CÁC CỘT DỄ DÍNH LỖI (DEEP DIVE)
# =========================================================================
print("🎯 BẮT ĐẦU SOI SÂU VÀO TỪNG CỘT CỤ THỂ...\n")

print("1. Tỷ lệ mất cân bằng nhãn AI (Fraud Labels):")
print(df_labels_raw['is_fraud'].value_counts(dropna=False, normalize=True) * 100)
print("-" * 60)

print("2. Lỗi định dạng rác trong phương thức quẹt thẻ (use_chip):")
print(df_trans_raw['use_chip'].value_counts(dropna=False))
print("-" * 60)

print("3. Tỷ lệ trống và các mã lỗi thực tế của hệ thống (errors):")
print(df_trans_raw['errors'].value_counts(dropna=False))
print("-" * 60)

print("4. Soi cấu trúc văn bản của cột số dư/thu nhập (để xem có dính dấu $ hoặc , không):")
print("Bảng Transactions - Cột 'amount' (3 dòng đầu):")
print(df_trans_raw['amount'].head(3).tolist())
print("\nBảng Users - Cột 'yearly_income' (3 dòng đầu):")
print(df_users_raw['yearly_income'].head(3).tolist())

🔍 BẮT ĐẦU CHỤP X-QUANG DỮ LIỆU THÔ (RAW DATA PROFILING)...

📊 BÁO CÁO DÒ LỖI: BẢNG TRANSACTIONS (THÔ)
🔹 Kích thước: 100000 dòng, 12 cột

⚠️ Lỗi Khuyết thiếu (NaN):
                Số dòng trống  Tỷ lệ (%)
merchant_state          11016     11.016
zip                     11484     11.484
errors                  98448     98.448

⚠️ Lỗi Trùng lặp (Duplicates):
👉 Số dòng bị trùng lặp hoàn toàn: 0

⚠️ Kiểu dữ liệu (Dtypes) đang bị nhận diện:
id                  int64
date                  str
client_id           int64
card_id             int64
amount                str
use_chip              str
merchant_id         int64
merchant_city         str
merchant_state        str
zip               float64
mcc                 int64
errors                str
dtype: object

------------------------------------------------------------

📊 BÁO CÁO DÒ LỖI: BẢNG USERS PROFILES (THÔ)
🔹 Kích thước: 2000 dòng, 14 cột

⚠️ Lỗi Khuyết thiếu (NaN):
👉 Sạch sẽ: Không có cột nào bị rỗng.

⚠️ Lỗi Trùng lặp (Duplicates

In [3]:
# =========================================================================
# 📊 CELL BỔ SUNG: DÒ NGƯỠNG OUTLIER CHUYÊN SÂU (DEEP-DIVE OUTLIERS)
# =========================================================================
print("🎯 BẮT ĐẦU SĂN TÌM GIÁ TRỊ NGOẠI LAI (OUTLIERS)...\n")

# 1. Khảo sát cột 'amount' ở bảng Transactions (Sample)
print("1. BẢNG TRANSACTIONS - CỘT 'AMOUNT'")
print("-" * 40)
# Tạm thời dọn sạch ký tự để ép kiểu tính toán
trans_amount_clean = df_trans_raw['amount'].astype(str).str.replace('$', '', regex=False).str.replace(',', '', regex=False).astype(float)

print("🔹 Thống kê mô tả tổng quan:")
print(trans_amount_clean.describe())
print("\n🔹 Chi tiết các mốc phân vị cao (Phát hiện điểm nhảy vọt):")
print(trans_amount_clean.quantile([0.90, 0.95, 0.98, 0.99, 0.995, 0.999]))
print("=" * 60 + "\n")


# 2. Khảo sát các cột tiền tệ ở bảng Users Profiles
print("2. BẢNG USERS PROFILES - CÁC CỘT THU NHẬP & NỢ")
print("-" * 40)
user_cols = ['per_capita_income', 'yearly_income', 'total_debt']
for col in user_cols:
    if col in df_users_raw.columns:
        user_col_clean = df_users_raw[col].astype(str).str.replace('$', '', regex=False).str.replace(',', '', regex=False).astype(float)
        print(f"🔹 Cột '{col}' - Ngưỡng cực đại (Max): {user_col_clean.max()}")
        print(f"🔹 Cột '{col}' - Mốc phân vị 99% (Ngưỡng thông thường): {user_col_clean.quantile(0.99)}")
        print(f"🔹 Cột '{col}' - Mốc phân vị 99.9%: {user_col_clean.quantile(0.999)}\n")
print("=" * 60 + "\n")


# 3. Khảo sát cột 'credit_limit' ở bảng Cards Profiles
print("3. BẢNG CARDS PROFILES - CỘT 'CREDIT_LIMIT'")
print("-" * 40)
if 'credit_limit' in df_cards_raw.columns:
    card_limit_clean = df_cards_raw['credit_limit'].astype(str).str.replace('$', '', regex=False).str.replace(',', '', regex=False).astype(float)
    print("🔹 Thống kê mô tả tổng quan:")
    print(card_limit_clean.describe())
    print("\n🔹 Mốc phân vị cao:")
    print(card_limit_clean.quantile([0.95, 0.99, 0.999]))

🎯 BẮT ĐẦU SĂN TÌM GIÁ TRỊ NGOẠI LAI (OUTLIERS)...

1. BẢNG TRANSACTIONS - CỘT 'AMOUNT'
----------------------------------------
🔹 Thống kê mô tả tổng quan:
count    100000.000000
mean         43.120798
std          82.952348
min        -500.000000
25%           8.860000
50%          28.940000
75%          64.560000
max        2807.050000
Name: amount, dtype: float64

🔹 Chi tiết các mốc phân vị cao (Phát hiện điểm nhảy vọt):
0.900    106.17200
0.950    145.55150
0.980    219.37040
0.990    318.00000
0.995    451.04370
0.999    874.20773
Name: amount, dtype: float64

2. BẢNG USERS PROFILES - CÁC CỘT THU NHẬP & NỢ
----------------------------------------
🔹 Cột 'per_capita_income' - Ngưỡng cực đại (Max): 163145.0
🔹 Cột 'per_capita_income' - Mốc phân vị 99% (Ngưỡng thông thường): 58537.75999999998
🔹 Cột 'per_capita_income' - Mốc phân vị 99.9%: 137441.15499999968

🔹 Cột 'yearly_income' - Ngưỡng cực đại (Max): 307018.0
🔹 Cột 'yearly_income' - Mốc phân vị 99% (Ngưỡng thông thường): 118866.4599